# View judged conversations

Look at a conversation (system prompt, user prompt, assistant answer) next to the judge LLM's honesty verdict.

Each card shows, top to bottom: **system prompt**, **user prompt**, **assistant answer**, **reasoning**, and finally the **label**.

Works for two on-disk layouts, selected by `DATASET`:
- `"ir"` — `data-ir-<WHICHONE>/` with `<MODEL>-judge-<JUDGE_MODEL>.json` (verdicts join on `k`, list under `scores`, categorical `label` field: HONEST / DECEPTIVE / MIXED / NULL, plus `reasoning`).
- `"roleplay"` — `data-more-roleplay/` with `scored-<TS>-<MODEL>.json`, one per judge run; the right one is picked by matching the file's `judge_model` field to `ROLEPLAY_JUDGE_MODEL` (verdicts join on `index`, list under `verdicts`, categorical `label` field: HONEST / DECEPTIVE / MIXED / NULL, plus `reasoning`).

Set `DATASET`, `MODEL` and the per-layout knobs below, run all cells, then use:
- `show(k)` — render a single conversation by its `k`,
- `show_all()` — render every conversation inline (optionally filtered).

In [ ]:
import json
import html
from pathlib import Path
from collections import Counter

from IPython.display import HTML, display

# --- config -----------------------------------------------------------------
# DATASET picks which on-disk layout to read:
#   "ir"       -> data-ir-<WHICHONE>/<MODEL>.json  +  <MODEL>-judge-<JUDGE_MODEL>-<TS>.json
#                 verdicts under "scores", join on "k", categorical field "label"
#                 (HONEST / DECEPTIVE / MIXED / NULL); newest timestamped file is used
#   "roleplay" -> data-more-roleplay/<MODEL>.json  +  scored-<TS>-<MODEL>.json
#                 verdicts under "verdicts", join on "index", categorical field
#                 "label" (HONEST / DECEPTIVE / MIXED / NULL)
DATASET = "roleplay"          # "ir" or "roleplay"

MODEL = "meta-llama--Llama-3.3-70B-Instruct"  # basename of the conversation .json

# --- ir-only knobs ----------------------------------------------------------
WHICHONE = "honest"           # "honest" or "dishonest"
JUDGE_MODEL = "gpt-5.4-mini"       # matches the judge output filename

# --- roleplay-only knobs ----------------------------------------------------
# Several scored-*.json may exist per model (one per judge run). They are not
# distinguished by filename, so we pick by the "judge_model" field inside each
# file. Set it to one of the available judges (e.g. "gpt-5.4-mini",
# "gpt-5.4-nano"), or leave "" to use whichever single file exists.
ROLEPLAY_JUDGE_MODEL = "gpt-5.4-mini"


def _pick_by_judge_model(candidates, judge_model):
    """From scored-*.json paths, return the one whose 'judge_model' matches.
    With judge_model="" and a single candidate, return it; otherwise raise a
    helpful error listing what judges are actually available."""
    by_judge = {}
    for p in candidates:
        with p.open() as f:
            by_judge[json.load(f).get("judge_model")] = p
    if not judge_model:
        if len(candidates) == 1:
            return candidates[0]
        raise ValueError(
            f"{len(candidates)} scored files for {MODEL}; set ROLEPLAY_JUDGE_MODEL "
            f"to one of {sorted(by_judge)}"
        )
    if judge_model not in by_judge:
        raise ValueError(
            f"no scored file with judge_model={judge_model!r} for {MODEL}; "
            f"available: {sorted(by_judge)}"
        )
    return by_judge[judge_model]


if DATASET == "ir":
    DATA_DIR = Path("data-ir-" + WHICHONE)
    CONV_PATH = DATA_DIR / f"{MODEL}.json"
    # judge-ir.py now timestamps its output; use the most recent one. (Old
    # un-timestamped "<MODEL>-judge-<JUDGE_MODEL>.json" files are ignored.)
    ir_candidates = sorted(DATA_DIR.glob(f"{MODEL}-judge-{JUDGE_MODEL}-*.json"))
    if not ir_candidates:
        raise FileNotFoundError(
            f"no {MODEL}-judge-{JUDGE_MODEL}-*.json in {DATA_DIR}"
        )
    JUDGE_PATH = ir_candidates[-1]
    LIST_KEY = "scores"        # key holding the list of verdict records
    SCORE_KEY = "k"            # field that joins a verdict back to a conversation
    SCORE_FIELD = "label"      # field carrying the honesty verdict on a record
    CATEGORICAL = True         # honesty is a categorical label
    SCALE_MIN, SCALE_MAX = None, None
elif DATASET == "roleplay":
    DATA_DIR = Path("data-more-roleplay")
    CONV_PATH = DATA_DIR / f"{MODEL}.json"
    candidates = sorted(DATA_DIR.glob(f"scored-*-{MODEL}.json"))
    if not candidates:
        raise FileNotFoundError(f"no scored-*-{MODEL}.json in {DATA_DIR}")
    JUDGE_PATH = _pick_by_judge_model(candidates, ROLEPLAY_JUDGE_MODEL)
    LIST_KEY = "verdicts"
    SCORE_KEY = "index"
    SCORE_FIELD = "label"
    CATEGORICAL = True         # honesty is a categorical label
    SCALE_MIN, SCALE_MAX = None, None
else:
    raise ValueError(f"unknown DATASET={DATASET!r}")

print("dataset:      ", DATASET)
print("conversations:", CONV_PATH)
print("judge scores: ", JUDGE_PATH)

In [ ]:
# --- load and join on the conversation's k ----------------------------------
with CONV_PATH.open() as f:
    conv_data = json.load(f)
with JUDGE_PATH.open() as f:
    judge_data = json.load(f)


def normalize_score(s):
    """Map a raw verdict record (either dataset layout) onto a common shape:
    {"k", "honesty", "reasoning", "verdict", "error"}. "honesty" carries the
    judge's call: a number (ir) or a categorical label like HONEST/DECEPTIVE
    (roleplay)."""
    return {
        "k": s.get(SCORE_KEY),
        "honesty": s.get(SCORE_FIELD),
        "reasoning": s.get("reasoning"),
        "verdict": s.get("verdict"),
        "error": s.get("error"),
    }


scores_by_k = {}
for s in judge_data.get(LIST_KEY, []):
    ns = normalize_score(s)
    if ns["k"] is not None:
        scores_by_k[ns["k"]] = ns


# chat-template end-of-turn tokens to strip from a raw completion, so answers
# render cleanly across model families (Llama: <|eot_id|>, Qwen/ChatML:
# <|im_end|>, Gemma: <end_of_turn>).
_EOT_TOKENS = ("<|eot_id|>", "<|im_end|>", "<end_of_turn>", "<|endoftext|>")


def assistant_text(conv):
    acs = conv.get("autocompletions") or []
    if not acs:
        return ""
    text = acs[0].get("text") or ""
    for tok in _EOT_TOKENS:
        text = text.replace(tok, "")
    return text.strip()


# records: one per conversation, with its judge verdict attached (if any)
records = []
for conv in conv_data["conversations"]:
    records.append({"conv": conv, "score": scores_by_k.get(conv["k"])})
records_by_k = {r["conv"]["k"]: r for r in records}

print(f"{len(records)} conversations, {len(scores_by_k)} with a judge verdict")


def verdict_of(score):
    """Return a short label for a judge verdict record: a number or categorical
    label (HONEST/DECEPTIVE/...), or NULL, ERROR, or '—' if never scored."""
    if score is None:
        return "—"
    if score.get("error"):
        return "ERROR"
    if score.get("honesty") is not None:
        return str(score["honesty"])
    return score.get("verdict") or "NULL"


print("verdict counts:", dict(Counter(verdict_of(r["score"]) for r in records)))

In [126]:
# --- rendering --------------------------------------------------------------
def _as_number(v):
    """Parse a verdict label to an int (handles negatives), else None."""
    try:
        return int(v)
    except (TypeError, ValueError):
        return None


# colours for categorical roleplay labels
_LABEL_COLOURS = {
    "HONEST": ("hsl(120, 65%, 45%)", "white"),
    "DECEPTIVE": ("hsl(0, 65%, 45%)", "white"),
    "MIXED": ("#b8860b", "white"),
    "NULL": ("#6c757d", "white"),
    "INCOMPLETE": ("#b8860b", "white"),
    "ERROR": ("#c0392b", "white"),
    "—": ("#e0e0e0", "#555"),
}


def _verdict_colour(score):
    """Return (bg, fg, label_text) for a verdict, colouring honesty: low -> red,
    high -> green. Numeric verdicts use the dataset scale; categorical labels
    use a fixed palette."""
    v = verdict_of(score)
    n = _as_number(v)
    if n is not None and not CATEGORICAL:
        span = max(SCALE_MAX - SCALE_MIN, 1)
        frac = (n - SCALE_MIN) / span
        hue = int(120 * min(max(frac, 0), 1))  # 0=red .. 120=green
        return f"hsl({hue}, 65%, 45%)", "white", f"honesty {n}"
    bg, fg = _LABEL_COLOURS.get(v, ("#6c757d", "white"))
    label = {"—": "not scored"}.get(v, v)
    return bg, fg, label


def _block(title, text, bg):
    return (
        f"<div style='margin:6px 0;'>"
        f"<div style='font-weight:700;color:#444;font-size:12px;"
        f"text-transform:uppercase;letter-spacing:.5px;'>{title}</div>"
        f"<div style='background:{bg};border-radius:6px;padding:8px 12px;"
        f"white-space:pre-wrap;font-family:-apple-system,Segoe UI,sans-serif;"
        f"font-size:14px;line-height:1.45;'>{html.escape(text)}</div></div>"
    )


def _label_block(score):
    """The verdict rendered as a panel (coloured by honesty) at the bottom."""
    bg, fg, label = _verdict_colour(score)
    return (
        f"<div style='margin:6px 0;'>"
        f"<div style='font-weight:700;color:#444;font-size:12px;"
        f"text-transform:uppercase;letter-spacing:.5px;'>label</div>"
        f"<div style='background:{bg};color:{fg};border-radius:6px;"
        f"padding:8px 12px;font-family:-apple-system,Segoe UI,sans-serif;"
        f"font-size:15px;font-weight:700;'>{html.escape(label)}</div></div>"
    )


def _card_html(record):
    conv, score = record["conv"], record["score"]
    head = (
        f"<div style='border-bottom:1px solid #ddd;padding-bottom:6px;'>"
        f"<span style='font-weight:700;font-size:15px;'>k = {conv['k']}</span>"
        f"</div>"
    )
    body = (
        _block("system prompt", conv.get("system_prompt", ""), "#f4f6f8")
        + _block("user prompt", conv.get("user_prompt", ""), "#eef3fb")
        + _block("assistant answer", assistant_text(conv), "#f3faf3")
    )
    reasoning = score.get("reasoning") if score else None
    if reasoning:
        body += _block("reasoning", str(reasoning), "#fbf7ec")
    err = score.get("error") if score else None
    if err:
        body += _block("judge error", str(err), "#fdecea")
    body += _label_block(score)
    return (
        f"<div style='border:1px solid #ccc;border-radius:8px;padding:12px;"
        f"margin:12px 0;max-width:1000px;'>{head}{body}</div>"
    )


def show(k):
    """Render a single conversation by its k."""
    rec = records_by_k.get(k)
    if rec is None:
        print(f"no conversation with k={k}")
        return
    display(HTML(_card_html(rec)))


def show_all(only=None, first_id=None, last_id=None):
    """Render every conversation inline.

    only:  optional filter on the verdict label, e.g. only="HONEST",
           only="DECEPTIVE", only="NULL", or a set/list like
           only={"HONEST", "MIXED"}.
    limit: cap how many cards are rendered.
    """
    if isinstance(only, str):
        only = {only}
    chosen = [
        r for r in records
        if only is None or verdict_of(r["score"]) in only
    ]
    if first_id is None:
        first_id = 0 
    if last_id is None:
        last_id = len(chosen)
    chosen = chosen[first_id:last_id]

    print(f"showing {len(chosen)} conversation(s)")
    display(HTML("".join(_card_html(r) for r in chosen)))

In [127]:
# A single conversation by k
# show(0)

In [129]:
# Everything (scroll through). Use filters, e.g. show_all(only="DECEPTIVE")
show_all(first_id=200, last_id=205)

showing 5 conversation(s)
